# Day 3: Procedural Combinational Logic

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File | Slides |
|---|---------|----------|------|--------|
| 1 | The `always @(*)` Block | ~12 min | `d03_s1_always_star_block.html` | 6 |
| 2 | `if/else` and `case` | ~15 min | `d03_s2_if_else_and_case.html` | 7 |
| 3 | The Latch Problem | ~12 min | `d03_s3_the_latch_problem.html` | 6 |
| 4 | Blocking vs. Nonblocking | ~6 min | `d03_s4_blocking_vs_nonblocking.html` | 7 |

## Code Examples

| File | Description | Synthesizable? |
|------|-------------|----------------|
| `code/day03_ex01_latch_demo.v` | Intentional latch — see Yosys warnings | Yes (with warnings) |
| `code/day03_ex02_latch_fixed.v` | Fixed version using default assignment | Yes (clean) |
| `code/day03_ex03_alu_4bit.v` | 4-bit ALU with case statement | Yes |

## Diagrams

| File | Used In | Description |
|------|---------|-------------|
| `diagrams/d03_latch_vs_comb.svg` | Seg 3 | Latch vs combinational side-by-side comparison |

## Pre-Class Quiz

See `day03_quiz.md` — 4 questions. Also embedded at end of Segment 4.

## Directory Structure

```
week1_day03/
├── day03_readme.md
├── day03_quiz.md
├── d03_s1_always_star_block.html
├── d03_s2_if_else_and_case.html
├── d03_s3_the_latch_problem.html
├── d03_s4_blocking_vs_nonblocking.html
├── code/
│   ├── day03_ex01_latch_demo.v
│   ├── day03_ex02_latch_fixed.v
│   └── day03_ex03_alu_4bit.v
└── diagrams/
    └── d03_latch_vs_comb.svg
```

---
## Code Examples

### `day03_ex01_latch_demo.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day03_ex01_latch_demo.v
// Course:  Accelerated HDL for Digital System Design — Day 3
//
// Description:
//   INTENTIONAL latch inference examples for educational purposes.
//   Synthesize with: yosys -p "synth_ice40 -top latch_demo" day03_ex01_latch_demo.v
//   Observe the latch warnings in Yosys output.
//   Then compare with day03_ex02_latch_fixed.v
//-----------------------------------------------------------------------------
module latch_demo (
    input  wire       i_sel,
    input  wire       i_enable,
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_opcode,
    output reg  [3:0] o_bug1,
    output reg  [3:0] o_bug2
);
    // BUG 1: Missing else — latch on o_bug1
    always @(*) begin
        if (i_sel)
            o_bug1 = i_a;
        // no else! → LATCH
    end

    // BUG 2: Incomplete case — latch on o_bug2
    always @(*) begin
        case (i_opcode)
            2'b00: o_bug2 = i_a;
            2'b01: o_bug2 = i_b;
            // 2'b10, 2'b11 missing → LATCH
        endcase
    end
endmodule
```

### `day03_ex02_latch_fixed.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day03_ex02_latch_fixed.v
// Course:  Accelerated HDL for Digital System Design — Day 3
//
// Description:
//   Fixed versions of the latch examples from day03_ex01_latch_demo.v.
//   Uses default assignment and default case to prevent latches.
//   Synthesize and verify: NO latch warnings.
//-----------------------------------------------------------------------------
module latch_fixed (
    input  wire       i_sel,
    input  wire       i_enable,
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_opcode,
    output reg  [3:0] o_fix1,
    output reg  [3:0] o_fix2
);
    // FIX 1: Default assignment at top
    always @(*) begin
        o_fix1 = 4'b0000;   // default: covers all paths
        if (i_sel)
            o_fix1 = i_a;
    end

    // FIX 2: Complete case with default
    always @(*) begin
        case (i_opcode)
            2'b00:   o_fix2 = i_a;
            2'b01:   o_fix2 = i_b;
            2'b10:   o_fix2 = i_a & i_b;
            2'b11:   o_fix2 = i_a | i_b;
            default: o_fix2 = 4'b0000;
        endcase
    end
endmodule
```

### `day03_ex03_alu_4bit.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day03_ex03_alu_4bit.v
// Course:  Accelerated HDL for Digital System Design — Day 3
// Board:   Nandland Go Board (Lattice iCE40 HX1K, VQ100)
//
// Description:
//   4-bit ALU demonstrating case statement pattern.
//   Operations: ADD, SUB, AND, OR selected by 2-bit opcode.
//   Includes zero flag and carry output.
//
// Build:
//   yosys -p "synth_ice40 -top alu_4bit -json day03_ex03_alu_4bit.json" day03_ex03_alu_4bit.v
//-----------------------------------------------------------------------------
module alu_4bit (
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_op,
    output reg  [3:0] o_result,
    output wire       o_zero,
    output reg        o_carry
);
    always @(*) begin
        o_carry  = 1'b0;       // default
        o_result = 4'b0000;    // default
        case (i_op)
            2'b00: {o_carry, o_result} = i_a + i_b;  // ADD
            2'b01: {o_carry, o_result} = i_a - i_b;  // SUB
            2'b10: o_result = i_a & i_b;              // AND
            2'b11: o_result = i_a | i_b;              // OR
            default: o_result = 4'b0000;
        endcase
    end

    assign o_zero = (o_result == 4'b0000);
endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** Why must you use `@(*)` instead of a manual sensitivity list?

<details><summary>Answer</summary>
Manual lists risk sim/synth mismatch. If you forget a signal, simulation won't update when it changes, but synthesis will. `@(*)` automatically includes all signals read inside the block.
</details>

**Q2:** What causes an unintentional latch? Give an example.

<details><summary>Answer</summary>
Not assigning a signal in every possible path through `always @(*)`. Example: `if (sel) y = a;` with no `else` — when `sel=0`, `y` must hold its value → latch inferred.
</details>

**Q3:** Name three techniques to prevent latch inference.

<details><summary>Answer</summary>
1. Default assignment at the top of the block
2. Complete `if/else` chains (always have an `else`)
3. `default` case in `case` statements
</details>

**Q4:** When do you use `=` vs `<=`?

<details><summary>Answer</summary>
`=` (blocking) for combinational `always @(*)`. `<=` (nonblocking) for sequential `always @(posedge clk)`. Never mix them in the same block.
</details>